In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
from datetime import datetime
import boto3

# constants
str_project = '20231010-gen-xii'
str_task = '13_payload_parsing'
str_subtask = 'concatenate_parsed_payloads'

# get today's date
str_date_today = datetime.today().strftime('%Y%m%d')

# init
cls_client = boto3.client('s3')

# prefix
str_prefix = f'{str_task}/days/{str_date_today}/parsed_payloads'

# list the files
dict_response = cls_client.list_objects_v2(
    Bucket=str_project,
    Prefix=str_prefix,
)

# get contents
list_dict_contents = dict_response['Contents']

# get the filenames
list_str_files = [dict_contents['Key'] for dict_contents in list_dict_contents]

# get only gzip
list_str_files = [str_file for str_file in list_str_files if '.gzip' in str_file]
print(f'There are {len(list_str_files)} parsed files:')
for a, str_file in enumerate(list_str_files):
    print(f'{a+1} - {str_file}')

# import and concatenate
list_df = []
for str_file in list_str_files:
    # read
    str_uri = f's3://{str_project}/{str_file}'
    df = pd.read_parquet(str_uri)
    # append
    list_df.append(df)
# concat
df = pd.concat(list_df)
# save memory
del list_df

# write to s3
str_filename = 'df_concat.gzip'
str_uri = f's3://{str_project}/{str_task}/days/{str_date_today}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-concat-parsed-payloads

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  26.11kB
Step 1/7 : FROM python:3.9
 ---> 4b15bb967077
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> c59ecdb5b59b
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 692397bbf273
Step 4/7 : COPY requirements.txt .
 ---> 1a1d58c975ff
Step 5/7 : RUN pip install -r requirements.txt
 ---> Running in 9da0f23c7381
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 87.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-west-2:836690756591:repository/genxii-concat-parsed-payloads",
        "registryId": "836690756591",
        "repositoryName": "genxii-concat-parsed-payloads",
        "repositoryUri": "836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-concat-parsed-payloads",
        "createdAt": 1710868890.804,
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": true
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-concat-parsed-payloads]
bb0f7401f91e: Preparing
39e166e24110: Preparing
4517dbe33629: Preparing
bf86e0cc03ba: Preparing
69b638ea12af: Preparing
afe28ac5c5d1: Preparing
f3b460831925: Preparing
20e2f78dadaf: Preparing
e077e19b6682: Preparing
21e1c4948146: Preparing
68866beb2ed2: Preparing
e6e2ab10dba6: Preparing
0238a1790

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass